# Harry Potter RAG — Lab 2: Evaluation Prep

Continuing from Lab 1, using *Harry Potter & The Philosopher's Stone* by J.K. Rowling.

Measuring retrieval or generation needs a labelled evaluation set first. This lab
builds one: it drafts questions from each chapter with a **third-party** model, then
locates every evidence snippet in the chunk index to attach the retrieval labels.

The output — `labs/eval/qa_candidates.jsonl` — is consumed directly by
[Lab 3: Evaluation](./lab3-rag-evaluation.ipynb), where the metrics and ablations live.

*Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published. AI coding agent is used in the process to assist notes writing and project planning.*

## Basic Setup (Reusing Lab 1)
- Import dependencies
- Setup env variables
- Setup Bedrock client
- Connect local chromadb

In [11]:
import boto3
from botocore.config import Config
import pymupdf
import re
import json
import os
import hashlib
from pathlib import Path
import tiktoken
import numpy as np
from tqdm.auto import tqdm
import chromadb
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
import csv

In [14]:
def repo_root() -> Path:
    """Walk up from the cwd until the repo root (the dir containing .git)."""
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p

ROOT = repo_root()
LABS = ROOT / "labs"
DATA = LABS / "data"

load_dotenv(dotenv_path=str(LABS / ".env"))

class Bedrock:
    """Bedrock helpers: Converse for chat, Titan for embeddings (sequential + cached)."""

    def __init__(self, model_id=None, region=None, embedding_model_id=None):
        self.region = region or os.getenv("AWS_REGION") or "ap-northeast-1"
        self.model_id = model_id or os.getenv("BEDROCK_CONVERSATION_MODEL_ID")
        if not self.model_id:
            raise ValueError("Set BEDROCK_CONVERSATION_MODEL_ID in .env")
        self.embedding_model_id = embedding_model_id or os.getenv("BEDROCK_EMBEDDING_MODEL_ID")
        if not self.embedding_model_id:
            raise ValueError("Set BEDROCK_EMBEDDING_MODEL_ID in .env")
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=self.region,
            config=Config(read_timeout=120, retries={"max_attempts": 5, "mode": "standard"}),
        )

    # ---- conversation (Q&A and judge): override model/params per call ----
    def chat(self, prompt, system=None, model_id=None, temperature=0.0, max_tokens=1024):
        kwargs = {
            "modelId": model_id or self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        response = self.client.converse(**kwargs)
        return response["output"]["message"]["content"][0]["text"]

    # ---- embeddings: sequential + disk cache ----
    @staticmethod
    def _cache_path():
        return DATA / "embeddings_cache.jsonl"

    @staticmethod
    def _cache_key(text, model_id, dimensions, normalize):
        raw = f"{model_id}|{dimensions}|{normalize}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _embed_one(self, text, dimensions, normalize):
        body = json.dumps({"inputText": text, "dimensions": dimensions, "normalize": normalize})
        resp = self.client.invoke_model(modelId=self.embedding_model_id, body=body)
        return json.loads(resp["body"].read())["embedding"]

    def embed(self, texts, dimensions=1024, normalize=True, cache_path=None):
        """Embed texts one by one, appending results to a JSONL cache.

        Returns an (n, dimensions) float32 array. 
        Re-running the notebook will proceed search for already-cached texts, 
        those will make no API calls.
        """
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        cache_path = Path(cache_path) if cache_path else self._cache_path()

        cache = {}
        if cache_path.exists():
            for line in cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["key"]] = rec["vector"]

        model = self.embedding_model_id
        keys = [self._cache_key(t, model, dimensions, normalize) for t in texts]
        todo = [(i, t) for i, t in enumerate(texts) if keys[i] not in cache]

        if todo:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "a", encoding="utf-8") as f:

                # Process bar
                for i, t in tqdm(todo, desc="embedding"):
                    vec = self._embed_one(t, dimensions, normalize)
                    cache[keys[i]] = vec
                    f.write(json.dumps({"key": keys[i], "vector": vec}) + "\n")

        return np.array([cache[k] for k in keys], dtype=np.float32)


bedrock_client = Bedrock()
judge_client = Bedrock(model_id=os.getenv("BEDROCK_JUDGE_MODEL_ID"))

chroma_client = chromadb.PersistentClient(path=str(LABS / ".chroma"))

## Goal of Lab 2: Evaluation Prep

Two independent systems, two independent failure modes — but neither can be measured
without questions whose evidence you can point at. So this lab builds the labelled set:

1. **Draft** — one prompt per chapter asks a third-party model (non-Amazon family, so
   its errors don't correlate with the generator under test) for grounded Q/A pairs,
   each carrying a **verbatim** evidence snippet.
2. **Locate** — match each snippet into the chunk index to produce the retrieval label
   (`evidence_chunk_ids`), scoped to the question's own chapter.
3. **Keep it draft-grade** — a snippet that can't be located is dropped, not trusted.

The set is used **directly** for evaluation: no gold-promotion, no hand review. See the
leakage note at the end for the one caveat that implies.

## Creating the candidate Q&A set

A set of Q&A to verify the quality of response from retrieval / generation:

- Using raw text per chapter as context, prompt a 3rd party LLM (non-Amazon family) to generate Q&A from each chapter
- Locate each evidence snippet in the chunk index to attach the retrieval labels
- Save as `.jsonl` and use directly for evaluation

In [6]:
DRAFTER_MODEL_ID = os.getenv(
    "BEDROCK_DRAFTER_MODEL_ID", "mistral.mistral-large-3-675b-instruct"
)
drafter = Bedrock(model_id=DRAFTER_MODEL_ID)

def ask_drafter(prompt, system=None, temperature=0.0, max_tokens=2048):
    """One turn against the drafter. Returns raw text."""
    return drafter.chat(
        prompt, system=system, temperature=temperature, max_tokens=max_tokens
    )

### Prompt set for Q&A generation

Every question carries a verbatim evidence span, we need candidates to be located in the chunk index and reviewed against the text rather than from memory.

A `type` param is required to show questions that require simple `factoid` context, or `multi-hop` multiple-context in same chapter. On the other hand, we set the `unanswerable` option as in context cannot be found from the retrieved context.

In [7]:
# --- Prompt set for grounded Q&A draft generation. ---
QA_SYSTEM = """You create ground-truth retrieval questions from a single chapter of a novel.

Each question must also carry `evidence`: the shortest span of the chapter that
contains the answer, copied CHARACTER-FOR-CHARACTER from the chapter text.

Hard rules:
- Copy `evidence` exactly as written. Never paraphrase, translate, fix spelling, or reorder words.
- The answer must be fully supported by the `evidence` span on its own.
- Prefer distinctive spans (names, objects, places, numbers) over generic phrasing.
- Never use a chapter heading or a page header as evidence.
- Keep `evidence` between 5 and 20 words.
- `type` is "factoid" (one fact) or "multi-hop" (two details from the chapter).
- Ask about specifics stated in this chapter, not general series trivia.
- Return strict JSON only: no prose, no markdown, no code fences.

Example
Chapter: "Millicent ran the shop on Bexley Street. The till held forty-two coins, all silver."
{"questions": [{"question": "How many coins were in the till?", "answer": "forty-two", "evidence": "The till held forty-two coins, all silver.", "type": "factoid"}]}"""

QA_USER = """{chapter_name}

<chapter>
{context}
</chapter>

Write {k} questions about this chapter. Return exactly:
{{"questions": [{{"question": "...", "answer": "...", "evidence": "...", "type": "factoid"}}]}}"""


def render_qa_prompt(chapter_name, context, k=4):
    """Fill the user template. Pass the chapter text as `context`, not the
    overlapping chunks (the 15% overlap repeats text at every boundary), and the
    value of `chapter_name` straight from the chunk metadata."""
    return QA_USER.format(chapter_name=chapter_name, context=context, k=k)

## Generate questions and answers

### Read chunk text

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

with open(DATA / "harrypotter-rawtext.json", "r", encoding="utf-8") as f:
    CHAPTER_TEXT = json.load(f)

CHAPTER_CHUNKS = read_jsonl(DATA / "harrypotter-chunks.jsonl")

CHAPTERS = sorted({
    (c["chapter_num"], c["chapter_name"]) 
    for c in CHAPTER_CHUNKS
    })


### Function to parse questions

In [ ]:
def parse_questions(text):
    """Model output -> list of question dicts, tolerating fences and stray prose."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    starts = [i for i in (text.find("["), text.find("{")) if i != -1]
    if not starts:
        raise ValueError("no JSON in model output")
    start, end = min(starts), max(text.rfind("]"), text.rfind("}"))
    data = json.loads(text[start:end + 1])
    return data if isinstance(data, list) else data.get("questions", [])

In [ ]:
# Loop through chapters and return question, answer and evidence (and type)

K_PER_CHAPTER = 4
CHAPTERS_TO_RUN = CHAPTERS

drafts = []
for chapter_num, chapter_name in CHAPTERS_TO_RUN:
    chapter_text = CHAPTER_TEXT[chapter_num - 1]
    raw = ask_drafter(
        render_qa_prompt(chapter_name, chapter_text, k=K_PER_CHAPTER),
        system=QA_SYSTEM,
    )
    questions = parse_questions(raw)

    for j, q in enumerate(questions, start=1):
        q.update(id=f"cand-hp1-{chapter_num:02d}-{j:02d}", book=1,
                 chapter=chapter_num, chapter_name=chapter_name, status="draft")
        drafts.append(q)
    print(f"ch{chapter_num:>2}: {len(questions)} drafted")

EVAL_DIR = LABS / "eval"
EVAL_DIR.mkdir(exist_ok=True)
out_path = EVAL_DIR / "qa_candidates.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for d in drafts:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
print(f"\nwrote {len(drafts)} candidates -> {out_path}")

In [16]:
# --- Locate each evidence snippet in the chunk index -> relevance labels. ---
# This is the only place a string is matched. Everything downstream scores on ids.

CHUNKS = read_jsonl(DATA / "harrypotter-chunks.jsonl")
CHUNKS_BY_CHAPTER = {}
for c in CHUNKS:
    CHUNKS_BY_CHAPTER.setdefault(c["chapter_num"], []).append(c)

# Fix encoding
_FOLD = str.maketrans({"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"',
                       "\u2014": "-", "\u2013": "-"})
def normalise(text):
    """One rule for both sides of a match: fold quotes/dashes, collapse whitespace, lowercase."""
    return re.sub(r"\s+", " ", text.translate(_FOLD)).strip().lower()
NORM_BY_CHUNK = {c["chunk_id"]: normalise(c["text"]) for c in CHUNKS}

SPLIT_RE = re.compile(r"\s*\[\.\.\.\]\s*")  # the drafter's joiner for multi-hop spans
MAX_MATCHES = 4
MIN_CHARS = 8

# Find evidence in chunks based on Q&A response
def locate_evidence(evidence, chapter_num):
    """Evidence text -> (chunk ids, per-span detail). Splits multi-hop spans on
    '[...]' and locates each part within the question's own chapter."""
    parts = [p for p in SPLIT_RE.split(evidence) if p.strip()] or [evidence]
    ids, spans = [], []
    for part in parts:
        needle = normalise(part).strip(" .,;:").strip('"').strip("'")
        hits = []
        if len(needle) >= MIN_CHARS:
            hits = [c["chunk_id"] for c in CHUNKS_BY_CHAPTER.get(chapter_num, [])
                    if needle in NORM_BY_CHUNK[c["chunk_id"]]]
        spans.append({"text": part, "chunk_ids": hits})
        ids.extend(h for h in hits if h not in ids)
    return ids, spans

# Load Q&A generated and match / evaluate, add new fields for jsonl
candidates = read_jsonl(EVAL_DIR / "qa_candidates.jsonl")
located = []
for r in candidates:
    ids, spans = locate_evidence(r.get("evidence", ""), r["chapter"])
    r["evidence_chunk_ids"] = ids
    r["n_spans"] = len(spans)
    r["locatable"] = bool(spans) and all(s["chunk_ids"] for s in spans)
    r["distinctive"] = r["locatable"] and all(1 <= len(s["chunk_ids"]) <= MAX_MATCHES for s in spans)
    r["unmatched_spans"] = [s["text"] for s in spans if not s["chunk_ids"]]
    located.append(r)

# Update original Q&A
with open(EVAL_DIR / "qa_candidates.jsonl", "w", encoding="utf-8") as f:
    for r in located:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


## Result from candidate Q&A generation

In [17]:
n_loc = sum(r["locatable"] for r in located)
n_dis = sum(r["distinctive"] for r in located)
print(f"{len(located)} candidates | locatable {n_loc} | distinctive {n_dis}")

per_ch = {}
for r in located:
    t, f = per_ch.get(r["chapter"], (0, 0))
    per_ch[r["chapter"]] = (t + 1, f + r["locatable"])
print("per chapter (located/total): " + "  ".join(
    f"ch{ch}:{f}/{t}" for ch, (t, f) in sorted(per_ch.items())))

miss = [r for r in located if not r["locatable"]]
print(f"not locatable ({len(miss)}):")
for r in miss:
    print(f"  {r['id']}: {' | '.join(r['unmatched_spans'])[:85]}")

69 candidates | locatable 65 | distinctive 65
per chapter (located/total): ch1:3/4  ch2:4/4  ch3:3/4  ch4:4/4  ch5:4/4  ch6:4/4  ch7:4/4  ch8:4/4  ch9:3/4  ch10:3/4  ch11:4/4  ch12:4/4  ch13:5/5  ch14:4/4  ch15:4/4  ch16:4/4  ch17:4/4
not locatable (4):
  cand-hp1-01-03: Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were 
  cand-hp1-03-03: he got out a hammer and nails and boarded up the mail slot.
  cand-hp1-09-01: He opened it excitedly and showed them a glass ball the size of a large marble, which
  cand-hp1-10-04: Ron


### Note: Not tested in this lab: parametric leakage

The candidate set is **auto-labelled** — each evidence span is located verbatim
in the chunk index — and is used **directly** as the evaluation set, with no
gold-promotion step.

One caveat belongs next to any *generation*-correctness number: the generator may
already know some answers from pretraining, which inflates absolute correctness.
It does **not** affect retrieval metrics, which score against the evidence chunk
ids, and it is roughly config-independent, so it largely cancels in the ablation
comparisons.